# Frozen Qwen3-Embedding-0.6B (2 classes)

**Question (Further Work item 3):** how does a model trained *to produce embeddings* compare with RoBERTa and ModernBERT?

Same code as `Frozen Qwen3-Embedding-0.6B.ipynb` (which says what the model is and how it's used), but on A2's original binary task: `rec.sport.baseball` vs. `rec.sport.hockey`, with the same data, split and classifier settings as `Frozen ModernBERT 2 class.ipynb`.

**Predict before running:** with these settings, RoBERTa got 91.63 (LR) and 92.92 (MLP) on test, and ModernBERT's `[CLS]` 74.39 / 76.32. Where will Qwen land?

## Load dataset (copied from A2)

In [1]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import fetch_20newsgroups

newsgroups_train = fetch_20newsgroups(
    subset='train', remove=('headers', 'footers', 'quotes'),
    categories=['rec.sport.baseball','rec.sport.hockey'])

newsgroups_test = fetch_20newsgroups(
    subset='test', remove=('headers', 'footers', 'quotes'),
    categories=['rec.sport.baseball','rec.sport.hockey'])

# Remove empty documents
for s, name in zip([newsgroups_train, newsgroups_test], ["train", "test"]):
  empty_indices = {i for i, doc in enumerate(s.data) if len(doc) == 0}
  orig_len = len(s.data)
  for k in ['data', 'filenames', 'target']:
    s[k] = [s[k][i] for i in range(orig_len) if i not in empty_indices]
  print(f"Removed {len(empty_indices)} empty documents from the {name} set. Before: {orig_len}. After: {len(s.data)}.")

y_train = np.array(newsgroups_train.target)
y_test = np.array(newsgroups_test.target)
n_classes = len(newsgroups_train.target_names)

Removed 26 empty documents from the train set. Before: 1197. After: 1171.
Removed 19 empty documents from the test set. Before: 796. After: 777.


## Qwen3-Embedding last-token embeddings

The same structure as the RoBERTa and ModernBERT notebooks, with three differences:
- **The instruction** goes in front of every message. It takes 17 tokens, so a message gets about 494 of the 512 tokens instead of 510.
- **Left padding:** the padding goes before short messages, so the last token of every message in a batch sits at position -1.
- **bfloat16**, the model's native precision (the vectors are converted to float32). Qwen's own examples also L2-normalize the vector, for cosine similarity; that's skipped here because the inputs are standardized below anyway.

The result is saved to `qwen3_0.6b_last_sports.npz`, so later runs skip this cell's slow part (about a minute on the GPU).

In [2]:
import os
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
CACHE = "qwen3_0.6b_last_sports.npz"
INSTRUCTION = "Instruct: Identify the topic or theme of the given news articles\nQuery:"

## X is a list of strings (messages). Returns a np 2D array of shape (len(X), 1024):
## the last-token embedding of each message, with the task instruction in front
def qwen_embed(X, tokenizer, qwen, batch_size=32):
    vecs = []
    for i in tqdm(range(0, len(X), batch_size)):
        batch = tokenizer([INSTRUCTION + x for x in X[i:i + batch_size]], padding=True, truncation=True,
                          max_length=512, return_tensors="pt").to(device)
        with torch.no_grad():  # frozen: no gradients needed
            out = qwen(**batch)
        # Padding is on the left, so position -1 is every message's last token (<|endoftext|>)
        vecs.append(out.last_hidden_state[:, -1].float().cpu().numpy())
    return np.concatenate(vecs)

if os.path.exists(CACHE):
    saved = np.load(CACHE)
    X_train, X_test = saved["X_train"], saved["X_test"]
else:
    # padding_side="left" so that every message's last token ends up at the same position, -1
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Embedding-0.6B", padding_side="left")
    # bfloat16 is the model's native precision and halves its memory
    qwen = AutoModel.from_pretrained("Qwen/Qwen3-Embedding-0.6B", torch_dtype=torch.bfloat16).to(device).eval()
    X_train = qwen_embed(newsgroups_train.data, tokenizer, qwen)
    X_test = qwen_embed(newsgroups_test.data, tokenizer, qwen)
    np.savez(CACHE, X_train=X_train, X_test=X_test)

print(X_train.shape, X_test.shape)

/home/tonyxdsu/venvs/cs423/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  0%|          | 0/37 [00:00<?, ?it/s]

  3%|▎         | 1/37 [00:02<01:17,  2.15s/it]

  5%|▌         | 2/37 [00:02<00:44,  1.27s/it]

  8%|▊         | 3/37 [00:03<00:33,  1.00it/s]

 11%|█         | 4/37 [00:04<00:28,  1.16it/s]

... (57 outputs left out)

 96%|█████████▌| 24/25 [00:16<00:00,  1.48it/s]

100%|██████████| 25/25 [00:16<00:00,  1.90it/s]

100%|██████████| 25/25 [00:16<00:00,  1.52it/s]

(1171, 1024) (777, 1024)


## Baseline: logistic regression

First, hold out 10% of train as a dev set, the same split as the RoBERTa notebooks. The MLP needs it for early stopping; the test set is only used at the very end. LR trains on the same 90%, so the comparison stays fair.

LR uses the RoBERTa search's best settings:
- **Standardized inputs** (mean 0 and std 1 in every dimension, from the 90%'s statistics only). Qwen's values spread about 51× wider than RoBERTa's `<s>` values (median per-dimension std 1.28 vs. 0.025). On raw inputs, the same `C` would mean a different amount of regularization. Standardized, both encoders are on the same scale, so RoBERTa's best `C` is a fair guess.
- **`C = 0.003`**, the best for standardized RoBERTa features on full 20 Newsgroups.
- **1 BLAS thread** (`threadpool_limits(1)`), which is about 60× faster on this machine.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
from threadpoolctl import threadpool_limits

X_tr, X_dev, y_tr, y_dev = train_test_split(
    X_train, y_train, test_size=0.1, stratify=y_train, random_state=423)

scaler = StandardScaler().fit(X_tr)  # mean and std of the 90% only, so dev and test stay unseen
X_tr, X_dev = scaler.transform(X_tr), scaler.transform(X_dev)

with threadpool_limits(1):
    logreg = LogisticRegression(C=0.003, max_iter=1000).fit(X_tr, y_tr)
lr_acc = metrics.accuracy_score(y_test, logreg.predict(scaler.transform(X_test)))
print(f"LR test accuracy: {lr_acc*100:.2f}")

LR test accuracy: 91.89


## MLP

1024 → 2048 → 64 → 2, the best RoBERTa config from the search (with Qwen's 1,024-d input instead of 768). The search found that these mattered:
- **A small learning rate:** AdamW at lr 1e-4. The first notebook's 1e-3 is too big a step once inputs are standardized. Weight decay stays at AdamW's default, 0.01.
- **Heavy dropout in every place:** 0.2 on the input, 0.7 after hidden layer 1, 0.5 after hidden layer 2. Without dropout, the MLP was only level with LR.
- **Label smoothing 0.1:** the training target puts 90% of the probability on the right class and spreads the other 10% evenly over all classes, so the model is never pushed toward 100% confidence.

GELU and batch size 32 also come from the search, but barely mattered there.

In [4]:
torch.manual_seed(423)

mlp = nn.Sequential(
    nn.Dropout(0.2),                                    # dropout on the input embedding
    nn.Linear(1024, 2048), nn.GELU(), nn.Dropout(0.7),  # hidden layer 1: wide, heavy dropout
    nn.Linear(2048, 64), nn.GELU(), nn.Dropout(0.5),    # hidden layer 2: narrow
    nn.Linear(64, n_classes),                           # one logit per class
).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(mlp.parameters(), lr=1e-4)  # weight decay: AdamW's default, 0.01

Training loop with early stopping, as in the A1 search: stop once dev loss hasn't improved for `PATIENCE` epochs, then go back to the best epoch's weights. Two changes from the first notebook, both from the search:
- **Patience 10 and up to 300 epochs** (was 5 and 100). A 10× smaller learning rate needs more epochs: at lr 1e-4, RoBERTa's best epoch on full 20 Newsgroups was around 50–90.
- **Dev loss is plain cross-entropy, without label smoothing.** The train loss includes smoothing, so it can't go below about 0.2 and isn't directly comparable to the dev loss.

In [5]:
import copy
import time
from torch.utils.data import TensorDataset, DataLoader

loader = DataLoader(TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr)), batch_size=32, shuffle=True)
X_dev_t, y_dev_t = torch.tensor(X_dev).to(device), torch.tensor(y_dev).to(device)

PATIENCE = 10
best = {"dev_loss": float("inf"), "epoch": 0, "state": None}
start = time.perf_counter()

for epoch in range(1, 301):
    mlp.train()  # dropout on
    train_loss = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(mlp(xb), yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(xb)

    mlp.eval()  # dropout off
    with torch.no_grad():
        dev_logits = mlp(X_dev_t)
        dev_loss = nn.functional.cross_entropy(dev_logits, y_dev_t).item()  # no label smoothing, as in the search
        dev_acc = (dev_logits.argmax(dim=1) == y_dev_t).float().mean().item()
    print(f"epoch {epoch:3d}  train loss {train_loss / len(X_tr):.3f}  dev loss {dev_loss:.3f}  dev acc {dev_acc*100:.2f}")

    if dev_loss < best["dev_loss"]:
        best = {"dev_loss": dev_loss, "epoch": epoch, "state": copy.deepcopy(mlp.state_dict())}
    elif epoch - best["epoch"] >= PATIENCE:
        break

mlp.load_state_dict(best["state"])
print(f"Training took {time.perf_counter() - start:.1f} s. Using epoch {best['epoch']} (lowest dev loss)")

epoch   1  train loss 0.430  dev loss 0.190  dev acc 91.53


epoch   2  train loss 0.333  dev loss 0.198  dev acc 91.53


epoch   3  train loss 0.322  dev loss 0.214  dev acc 90.68


epoch   4  train loss 0.311  dev loss 0.206  dev acc 91.53


epoch   5  train loss 0.305  dev loss 0.218  dev acc 90.68


epoch   6  train loss 0.296  dev loss 0.218  dev acc 92.37


epoch   7  train loss 0.289  dev loss 0.218  dev acc 92.37


epoch   8  train loss 0.284  dev loss 0.226  dev acc 91.53


epoch   9  train loss 0.274  dev loss 0.227  dev acc 89.83


epoch  10  train loss 0.278  dev loss 0.206  dev acc 91.53


epoch  11  train loss 0.276  dev loss 0.221  dev acc 89.83
Training took 3.4 s. Using epoch 1 (lowest dev loss)


## LR vs. MLP on the test set

In [6]:
mlp.eval()
with torch.no_grad():
    mlp_pred = mlp(torch.tensor(scaler.transform(X_test)).to(device)).argmax(dim=1).cpu().numpy()
mlp_acc = metrics.accuracy_score(y_test, mlp_pred)

print(f"LR  test accuracy: {lr_acc*100:.2f}")
print(f"MLP test accuracy: {mlp_acc*100:.2f}")
print(f"Difference: {(mlp_acc - lr_acc)*100:+.2f} points (1 point = {len(y_test) / 100:.0f} test messages)")

LR  test accuracy: 91.89
MLP test accuracy: 91.38
Difference: -0.51 points (1 point = 8 test messages)


For comparison, with the same settings (test accuracy):

| | LR | MLP |
|---|---|---|
| RoBERTa `<s>` (`Frozen ModernBERT 2 class.ipynb`'s code on `roberta_cls_sports.npz`) | 91.63 | 92.92 |
| ModernBERT `[CLS]` (`Frozen ModernBERT 2 class.ipynb`) | 74.39 | 76.32 |

This is one MLP with one seed, and the test set has only 777 messages (1 point ≈ 8 messages). A gap of a point or two can come from chance alone, so only larger gaps mean much. The dev set (10% of train) is also small, about 120 messages, so early stopping is noisier here.